# Procesor odmian — format TEI app/rdg

Uruchamiaj komórki kolejno (`Shift+Enter`).

**Potrzebne pliki:**
- `odmiany.txt` — rejestr odmian
- plik z tekstem wejściowym
- `config.txt` — mapowanie nazw przekazów na formy kanoniczne

**Format odmiany.txt:**
```
w. 1 tekst bazowy] wariant Swiadek
w. 4 tekst] wariant1 Swiadek1] wariant2 Swiadek2, Swiadek3
```

**Format wyjscia:**
```xml
tekst bazowy<app>
	<rdg wit="Tekst edycji">tekst bazowy</rdg>
	<rdg wit="Swiadek">wariant</rdg>
</app> dalszy tekst
```

## 1 — Ładowanie kodu

In [ ]:
import re
import io
from google.colab import files


def load_config(text):
    config = {}
    for line in text.splitlines():
        line = line.strip()
        if not line or line.startswith('#'):
            continue
        if '=' in line:
            k, v = line.split('=', 1)
            config[k.strip()] = v.strip()
    return config


def parse_variant_group(text, config):
    text = text.rstrip(',;').strip()
    all_names = set(config.keys())
    name_pat = '(?:' + '|'.join(
    re.escape(n) for n in sorted(all_names, key=len, reverse=True)
    ) + ')'
    pattern = r'\s+(' + name_pat + r'(?:\s*,\s*' + name_pat + r')*)\s*$'
    m = re.search(pattern, text)
    if not m:
        return text.strip(), []
    variant_text = text[:m.start()].strip()
    witnesses = []
    for w in re.split(r'\s*,\s*', m.group(1)):
        norm = config.get(w.strip())
        if norm and norm not in witnesses:
            witnesses.append(norm)
    return variant_text, witnesses


def parse_odmiany(text, config):
    text = text.replace('...', '\u2026')
    entry_pat = re.compile(
        r'^(?:s\.\s+[\d\-\u2013\u2014]+\s+)?w\.\s+[\d\-\u2013\u2014]+\s+'
    )
    match_pat = re.compile(
        r'^(?:s\.\s+[\d\-\u2013\u2014]+\s+)?w\.\s+([\d\-\u2013\u2014]+)\s+(.*?)\]\s*(.+)$'
    )
    joined = []
    current = None
    for raw in text.splitlines():
        line = raw.strip()
        if not line:
            if current:
                joined.append(current)
                current = None
            continue
        if entry_pat.match(line):
            if current:
                joined.append(current)
            current = line
        elif current:
            current = current + ' ' + line
    if current:
        joined.append(current)

    entries = []
    for line in joined:
        m = match_pat.match(line)
        if not m:
            continue
        nr_wersu = m.group(1)
        tekst_edycji    = m.group(2).strip()
        reszta    = m.group(3).strip()
        odmiany = []
        for g in re.split(r'[;\]]', reszta):
            g = g.strip()
            if not g:
                continue
            vtext, wits = parse_variant_group(g, config)
            if wits:
                odmiany.append((vtext, wits))
        if odmiany:
            entries.append({'nr_wersu': nr_wersu, 'tekst_edycji': tekst_edycji, 'odmiany': odmiany})
        else:
            print('UWAGA: w. ' + nr_wersu + ' -- nie rozpoznano przekazów: ' + tekst_edycji)
    return entries


def build_app(tekst_edycji, odmiany):
    lines = ['<app>']
    lines.append('\t<rdg wit="Tekst edycji">' + tekst_edycji + '</rdg>')
    for vtext, wits in odmiany:
        wit_str = ', '.join(wits)
        lines.append('\t<rdg wit="' + wit_str + '">' + vtext + '</rdg>')
    lines.append('</app>')
    return '\n'.join(lines)


def process(config_text, odmiany_text, input_text):
    config = load_config(config_text)

    entries = parse_odmiany(odmiany_text, config)
    original = input_text
    text_out = original
    report   = []

    for entry in entries:
        tekst_edycji     = entry['tekst_edycji']
        nr_wersu  = entry['nr_wersu']
        odmiany = entry['odmiany']

        occ = original.count(tekst_edycji)
        if occ == 0:
            report.append('NIEZNALEZIONY w. ' + nr_wersu + ': ' + tekst_edycji)
            continue
        if occ > 1:
            report.append('DUPLIKAT w. ' + nr_wersu + ': ' + tekst_edycji
                          + ' (' + str(occ) + ' wystąpień, oznaczono wszystkie)')

        app_block = build_app(tekst_edycji, odmiany)
        text_out = text_out.replace(tekst_edycji, tekst_edycji + app_block)

    return text_out, report


print('Kod załadowany! Przejdź do komorki 2.')


## 2 — Wgraj pliki i uruchom

In [ ]:
print("Wgraj odmiany.txt:")
up = files.upload()
odmiany_text = list(up.values())[0].decode('utf-8')

print("\nWgraj plik z tekstem wejsciowym:")
up = files.upload()
input_text = list(up.values())[0].decode('utf-8')

print("\nWgraj config.txt:")
up = files.upload()
config_text = list(up.values())[0].decode('utf-8')

text_out, report = process(config_text, odmiany_text, input_text)

print("\n=== RAPORT ===")
if report:
    for r in report:
        print(r)
else:
    print("Brak problemów — wszystkie warianty znalezione.")


## 3 — Pobierz wynik

In [ ]:
import os
os.makedirs('output', exist_ok=True)
with open('output/output.txt', 'w', encoding='utf-8') as f:
    f.write(text_out)
files.download('output/output.txt')
print("Pobieranie zakonczone!")
